# 🇨🇭 Swiss Legal Retrieval — Corpus Indexing

Builds a searchable index of Swiss federal laws and court decision
considerations for the [LLM Agentic Legal Retrieval competition](https://www.kaggle.com/competitions/llm-agentic-legal-information-retrieval).
Output: a `corpus.parquet` with about 2.9M chunks + bge-m3 embeddings
(173k from `laws_de.csv`, 2.74M from `court_considerations.csv`), plus
the corpus itself as the downstream artifact.

**This is a utility notebook** — attach it to your downstream notebooks via
`Add Input`. All the expensive work (3M chunks encoded on dual T4 GPU)
happens once, here.

## 🎯 What it does
1. 📦 Installs packages from the wheels utility notebook
2. 📂 Loads `laws_de.csv` (176k rows) + `court_considerations.csv` (2.5M rows)
3. 🧹 Drops parser junk and dispositif boilerplate
4. ✂️ Chunks long docs (different strategy per file)
5. ⚡ Encodes with bge-m3 in fp16 on dual T4
6. 💾 Writes `corpus.parquet`

## 💡 Key choices
- **bge-m3** — multilingual, handles EN↔DE queries, strong on MTEB
- **fp16** — 4.6× faster than fp32 on T4, no measurable quality loss
- **Dual chunking strategy** — paragraph-packed for laws, positional for decisions

## 👋 About me

Publishing this as a community hobby project — I'm **not eligible for the
prize pool**. Feel free to fork, modify, learn from it.

**I'm an Agentic AI Developer.** I work across the whole modern LLM
application stack.

**I'm looking for work** 🔍. If you're hiring for Agentic AI / LLM
application engineering roles, please reach out:

📧 **re.azami@gmail.com**

Remote, hybrid, or on-site — all welcome.

## 📦 Install packages

Install sentence-transformers from the wheels utility notebook. No internet
needed — everything comes from pre-downloaded wheels.

In [1]:
import sys, subprocess, os
from pathlib import Path

WHEELS_UTIL = Path("/kaggle/usr/lib/notebooks/rezaazami/utility_swiss_legal_wheels_and_models")
assert WHEELS_UTIL.exists(), f"Wheels notebook not attached at {WHEELS_UTIL}"

WHEELS = WHEELS_UTIL / "wheels"
MODEL_PATH = WHEELS_UTIL / "models" / "bge-m3"

for p in [WHEELS, MODEL_PATH]:
    assert p.exists(), f"Missing: {p}"

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--no-index", "--find-links", str(WHEELS),
    "sentence-transformers==5.4.0",
])
print("Packages installed from wheels")
print(f"Model path: {MODEL_PATH}")

Packages installed from wheels
Model path: /kaggle/usr/lib/notebooks/rezaazami/utility_swiss_legal_wheels_and_models/models/bge-m3


## 📂 Loading the corpus

Two CSV files from the competition:
- **`laws_de.csv`** — 176k Swiss statute articles (`Art. 963 Abs. 1 ZGB`, etc.)
- **`court_considerations.csv`** — 2.5M court decision snippets

Always look at the data before deciding anything. 👀

In [2]:
import pandas as pd

DATA = Path("/kaggle/input/competitions/llm-agentic-legal-information-retrieval")

laws = pd.read_csv(DATA / "laws_de.csv")
decisions = pd.read_csv(DATA / "court_considerations.csv")

print(f"Laws:      {len(laws):>10,} rows, columns: {list(laws.columns)}")
print(f"Decisions: {len(decisions):>10,} rows, columns: {list(decisions.columns)}")
print(f"\nLaws sample:")
print(laws.head(2).to_string())
print(f"\nDecisions sample:")
print(decisions.head(2).to_string())

Laws:         175,933 rows, columns: ['citation', 'text', 'title']
Decisions:  2,476,315 rows, columns: ['citation', 'text']

Laws sample:
     citation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

## 🧹 Cleaning, filtering, chunking

This is the most important cell. Three steps, each backed by data we
inspected carefully:

### Filtering out noise

**In decisions**, about 198k rows (8%) are dispositif boilerplate — formal
one-line rulings repeated thousands of times across unrelated cases:
`"Die Beschwerde wird abgewiesen."` × 15,912, etc. All under 50 chars.
**`DEC_MIN_CHARS = 50`** drops them cleanly.

**In laws**, about 4k rows (2%) are parser artifacts: `'Aufgehoben'`
(repealed markers), `'…4'` truncation artifacts, cross-references with no
content. **`LAWS_MIN_CHARS = 15`** with pattern filters.

### Chunking long documents

Laws have clean paragraph structure (newlines), but decisions have **zero
newlines** — stored as unbroken text walls. So:
- **Laws** → greedy-pack paragraphs on `\n` into ~2000-char chunks
- **Decisions** → positional sliding window, 200 char overlap

### Output

`records` list: ~2.9M dicts with `citation, text, title, embed_text,
source_type, chunk_id`.

In [3]:
import gc

CHUNK_SIZE = 2000
OVERLAP = 200
LAWS_MIN_CHARS = 15
DEC_MIN_CHARS = 50

def chunk_by_paragraphs(text, max_chars=CHUNK_SIZE):
    """Greedy-pack paragraphs on \\n boundaries. For laws."""
    paragraphs = text.split("\n")
    chunks, current = [], ""
    for p in paragraphs:
        p = p.strip()
        if not p:
            continue
        if current and len(current) + len(p) + 1 > max_chars:
            chunks.append(current)
            current = p
        else:
            current = current + "\n" + p if current else p
    if current:
        chunks.append(current)
    return chunks if chunks else [text]

def chunk_by_position(text, max_chars=CHUNK_SIZE, overlap=OVERLAP):
    """Positional sliding window. For decisions (no paragraph structure)."""
    if len(text) <= max_chars:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + max_chars
        chunks.append(text[start:end])
        start += max_chars - overlap
    return chunks

records = []

# --- Process laws ---
laws_kept = 0
for _, row in laws.iterrows():
    text = str(row["text"]).strip()
    citation = str(row["citation"]).strip()
    title = str(row.get("title", "")).strip()
    
    if len(text) < LAWS_MIN_CHARS:
        continue
    if "aufgehoben" in text.lower() and len(text) < 50:
        continue
    if "…" in text and len(text) < 30:
        continue
    
    chunks = chunk_by_paragraphs(text)
    for i, chunk in enumerate(chunks):
        records.append({
            "citation":    citation,
            "text":        chunk,
            "title":       title,
            "embed_text":  (title + " " + chunk).strip(),
            "source_type": "law",
            "chunk_id":    i,
        })
    laws_kept += 1

print(f"Laws: {len(laws):,} → {laws_kept:,} kept → {sum(1 for r in records if r['source_type']=='law'):,} chunks")

# --- Process decisions ---
dec_start = len(records)
dec_kept = 0
for _, row in decisions.iterrows():
    text = str(row["text"]).strip()
    citation = str(row["citation"]).strip()
    
    if len(text) < DEC_MIN_CHARS:
        continue
    
    chunks = chunk_by_position(text)
    for i, chunk in enumerate(chunks):
        records.append({
            "citation":    citation,
            "text":        chunk,
            "title":       "",
            "embed_text":  chunk,
            "source_type": "decision",
            "chunk_id":    i,
        })
    dec_kept += 1

dec_chunks = len(records) - dec_start
print(f"Decisions: {len(decisions):,} → {dec_kept:,} kept → {dec_chunks:,} chunks")
print(f"\nTotal records: {len(records):,}")

# Free the DataFrames — not needed anymore
del laws, decisions
gc.collect()

Laws: 175,933 → 173,264 kept → 174,090 chunks
Decisions: 2,476,315 → 2,278,693 kept → 2,755,052 chunks

Total records: 2,929,142


0

## ⚡ Encoding with fp16 on dual T4

The long cell (~4.5 hours). Key tricks:
- **`embedder.half()`** → fp16 on GPU, 4.6× faster than fp32
- **`encode_multi_process`** → both T4s in parallel (~1.7× extra)
- **Shard checkpointing** → crash at hour 4 loses 15 min, not everything
- **Cleanup at the end** → free model + pool + texts before Cell 5

In [4]:
import torch, time, shutil, gc, numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

assert torch.cuda.device_count() >= 2, "Set Accelerator → T4 x2 in Settings"

CHECKPOINT_DIR = Path("/kaggle/working/embedding_checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
SHARD_SIZE = 200_000

texts = [r["embed_text"] for r in records]
total = len(texts)
n_shards = (total + SHARD_SIZE - 1) // SHARD_SIZE
print(f"Total chunks: {total:,}  →  {n_shards} shards of {SHARD_SIZE:,}")

embedder = SentenceTransformer(str(MODEL_PATH))
embedder.max_seq_length = 1024
embedder.half()

pool = embedder.start_multi_process_pool(target_devices=["cuda:0", "cuda:1"])

try:
    all_shards = []
    t_start = time.time()
    
    for shard_idx, shard_start in enumerate(range(0, total, SHARD_SIZE)):
        shard_path = CHECKPOINT_DIR / f"shard_{shard_idx:03d}.npy"
        shard_end = min(shard_start + SHARD_SIZE, total)
        
        if shard_path.exists():
            print(f"[{shard_idx+1}/{n_shards}] Loading checkpoint: {shard_path.name}")
            shard_vecs = np.load(shard_path)
        else:
            print(f"[{shard_idx+1}/{n_shards}] Encoding {shard_start:,}:{shard_end:,} ...")
            t0 = time.time()
            shard_vecs = embedder.encode_multi_process(
                texts[shard_start:shard_end],
                pool,
                batch_size=64,
                normalize_embeddings=True,
            ).astype(np.float16)
            np.save(shard_path, shard_vecs)
            
            elapsed  = time.time() - t0
            total_el = time.time() - t_start
            done     = shard_end
            eta_min  = (total - done) * (total_el / done) / 60 if done > 0 else 0
            print(f"    done in {elapsed/60:.1f} min  "
                  f"| total: {total_el/60:.1f} min  "
                  f"| ETA: {eta_min:.1f} min")
        
        all_shards.append(shard_vecs)
    
    embeddings = np.concatenate(all_shards, axis=0)
    del all_shards
    
    print(f"\nFinal embeddings shape: {embeddings.shape}")
    print(f"dtype: {embeddings.dtype}")
    print(f"Size in memory: {embeddings.nbytes / 1e9:.2f} GB")
    print(f"Total encoding time: {(time.time() - t_start)/60:.1f} min")
    
    shutil.rmtree(CHECKPOINT_DIR)
    print("Removed shard checkpoints")
    
    del texts
    print("Freed texts list")

finally:
    embedder.stop_multi_process_pool(pool)
    del embedder
    del pool
    gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass
    print("Freed embedder, pool, CUDA cache")

import psutil
mem = psutil.virtual_memory()
print(f"\nRAM after cleanup: {mem.used/1e9:.1f} GB used / "
      f"{mem.total/1e9:.1f} GB total ({mem.percent:.0f}%)")

Total chunks: 2,929,142  →  15 shards of 200,000


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[1/15] Encoding 0:200,000 ...


/tmp/ipykernel_23/3934095018.py:36: DeprecationWarning: The `encode_multi_process` method has been deprecated, and its functionality has been integrated into `encode`. You can now call `encode` with the same parameters to achieve multi-process encoding.
  shard_vecs = embedder.encode_multi_process(


    done in 9.8 min  | total: 9.8 min  | ETA: 134.0 min
[2/15] Encoding 200,000:400,000 ...
    done in 20.6 min  | total: 30.4 min  | ETA: 192.0 min
[3/15] Encoding 400,000:600,000 ...
    done in 17.2 min  | total: 47.6 min  | ETA: 184.6 min
[4/15] Encoding 600,000:800,000 ...
    done in 17.6 min  | total: 65.2 min  | ETA: 173.4 min
[5/15] Encoding 800,000:1,000,000 ...
    done in 17.5 min  | total: 82.7 min  | ETA: 159.5 min
[6/15] Encoding 1,000,000:1,200,000 ...
    done in 17.3 min  | total: 100.0 min  | ETA: 144.0 min
[7/15] Encoding 1,200,000:1,400,000 ...
    done in 17.5 min  | total: 117.5 min  | ETA: 128.3 min
[8/15] Encoding 1,400,000:1,600,000 ...
    done in 17.9 min  | total: 135.4 min  | ETA: 112.5 min
[9/15] Encoding 1,600,000:1,800,000 ...
    done in 21.9 min  | total: 157.3 min  | ETA: 98.7 min
[10/15] Encoding 1,800,000:2,000,000 ...
    done in 21.2 min  | total: 178.5 min  | ETA: 82.9 min
[11/15] Encoding 2,000,000:2,200,000 ...
    done in 22.1 min  | total: 

## 💾 Writing corpus.parquet

Streaming write in 100k-row batches to avoid OOM. Vectors stay fp16
throughout — no fp32 cast. Compressed with zstd.

### Why streaming instead of one big `to_parquet`

The first time I ran this pipeline, the naive approach OOMed 73 seconds
after encoding finished. The list comprehension + fp32 cast peaked at ~28 GB
out of 33.7 GB available. The streaming approach peaks at ~13 GB with 17 GB
of headroom.

In [5]:
import gc, os
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import psutil

mem = psutil.virtual_memory()
print(f"RAM at start: {mem.used/1e9:.1f} GB / {mem.total/1e9:.1f} GB ({mem.percent:.0f}%)")

OUT = "/kaggle/working/corpus.parquet"
CHUNK = 100_000

schema = pa.schema([
    ("citation",    pa.string()),
    ("text",        pa.string()),
    ("title",       pa.string()),
    ("source_type", pa.string()),
    ("chunk_id",    pa.int32()),
    ("vector",      pa.list_(pa.float16(), 1024)),
])

total = len(records)
print(f"\nStreaming {total:,} rows to {OUT} in batches of {CHUNK:,}")
print(f"Batches: {(total + CHUNK - 1) // CHUNK}")

with pq.ParquetWriter(OUT, schema, compression="zstd") as writer:
    for start in range(0, total, CHUNK):
        end = min(start + CHUNK, total)
        
        batch_df = pd.DataFrame({
            "citation":    [records[i]["citation"]    for i in range(start, end)],
            "text":        [records[i]["text"]        for i in range(start, end)],
            "title":       [records[i]["title"]       for i in range(start, end)],
            "source_type": [records[i]["source_type"] for i in range(start, end)],
            "chunk_id":    [records[i]["chunk_id"]    for i in range(start, end)],
            "vector":      list(embeddings[start:end]),
        })
        
        table = pa.Table.from_pandas(batch_df, schema=schema, preserve_index=False)
        writer.write_table(table)
        
        del batch_df, table
        if (start // CHUNK) % 5 == 0:
            gc.collect()
            print(f"  {end:,} / {total:,}")

size_gb = os.path.getsize(OUT) / 1e9
print(f"\nWrote {OUT}: {size_gb:.2f} GB, {total:,} rows")

mem = psutil.virtual_memory()
print(f"RAM after write: {mem.used/1e9:.1f} GB / {mem.total/1e9:.1f} GB ({mem.percent:.0f}%)")

RAM at start: 15.6 GB / 33.7 GB (48%)

Streaming 2,929,142 rows to /kaggle/working/corpus.parquet in batches of 100,000
Batches: 30
  100,000 / 2,929,142
  600,000 / 2,929,142
  1,100,000 / 2,929,142
  1,600,000 / 2,929,142
  2,100,000 / 2,929,142
  2,600,000 / 2,929,142

Wrote /kaggle/working/corpus.parquet: 6.45 GB, 2,929,142 rows
RAM after write: 16.9 GB / 33.7 GB (52%)


## ✅ Verify output

Confirm the parquet is loadable and the output fits under the 19.5 GB limit.

In [6]:
import os, pandas as pd, numpy as np

# Verify parquet round-trips
print("Verifying parquet...")
test_df = pd.read_parquet("/kaggle/working/corpus.parquet", columns=["citation", "source_type"])
print(f"  Rows: {len(test_df):,}")
print(f"  Laws: {(test_df['source_type']=='law').sum():,}")
print(f"  Decisions: {(test_df['source_type']=='decision').sum():,}")

vec_sample = pd.read_parquet("/kaggle/working/corpus.parquet", columns=["vector"]).iloc[0]["vector"]
print(f"  Vector dtype: {np.asarray(vec_sample).dtype}, length: {len(vec_sample)}")

# Output directory
print("\nOutput contents:")
DST = "/kaggle/working"
for entry in sorted(os.listdir(DST)):
    path = os.path.join(DST, entry)
    if os.path.isdir(path):
        size = sum(os.path.getsize(os.path.join(dp, f))
                   for dp, _, fs in os.walk(path) for f in fs)
        print(f"  {entry}/  ({size/1e9:.2f} GB)")
    else:
        print(f"  {entry}  ({os.path.getsize(path)/1e9:.2f} GB)")

total = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, fs in os.walk(DST) for f in fs
)
print(f"\nTotal: {total/1e9:.2f} GB / 19.5 GB limit")

Verifying parquet...
  Rows: 2,929,142
  Laws: 174,090
  Decisions: 2,755,052
  Vector dtype: float16, length: 1024

Output contents:
  __notebook__.ipynb  (0.00 GB)
  corpus.parquet  (6.45 GB)

Total: 6.45 GB / 19.5 GB limit
